# Modelo de prioridad/riesgo — baseline vs. ML
### Factored AI & Data Hackathon 2026 — Equipo DataMastersGT

Este notebook documenta **todo el proceso**, incluyendo el camino que no funcionó, porque es parte
de la evidencia de buena práctica de datos/ML que pide el reto (no solo el resultado final).


## Paso 1 — Lo que intentamos primero (y por qué no funcionó)

La hipótesis inicial, basada en el EDA, era: **el banco incumple SLA ~20-22% de las veces sin importar
la prioridad asignada → entrenamos un modelo que prediga mejor el incumplimiento (`sla_breached`) que
esa prioridad actual.**

Antes de construir nada "bonito", probamos si existía señal real entre las features disponibles
(monto, categoría, canal, reincidencia, segmento, score de crédito) y los targets candidatos
(`sla_breached`, `was_escalated`, incluso la propia `priority` que asigna el banco).


In [1]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, roc_auc_score, classification_report

pd.set_option("display.max_columns", None)

con = duckdb.connect()
base = "../hackathon-data"
con.execute(f"CREATE VIEW complaints AS SELECT * FROM read_csv_auto('{base}/complaints/**/*.csv', ignore_errors=true, union_by_name=true)")
con.execute(f"CREATE VIEW customers AS SELECT * FROM read_csv_auto('{base}/customers.csv', ignore_errors=true)")

df = con.execute('''
    SELECT c.complaint_id, c.creation_date, c.customer_id, c.case_type, c.category, c.subcategory,
           c.reception_channel, c.claimed_amount, c.priority, c.is_repeat_complainer,
           cu.segment, cu.country, cu.credit_score, cu.estimated_monthly_income, c.sla_breached
    FROM complaints c LEFT JOIN customers cu ON c.customer_id = cu.customer_id
''').df()
df["creation_date"] = pd.to_datetime(df["creation_date"])
df = df.sort_values("creation_date").reset_index(drop=True)
df["sla_breached"] = df["sla_breached"].astype(int)
df["claimed_amount_missing"] = df["claimed_amount"].isna().astype(int)
print(df.shape)

(10332, 16)


In [2]:
# Prueba 1: ¿la prioridad que el banco ya asigna predice el incumplimiento de SLA?
prio_map = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3}
auc_priority_vs_breach = roc_auc_score(df["sla_breached"], df["priority"].map(prio_map))
print("AUC prioridad actual -> sla_breached:", round(auc_priority_vs_breach, 3), "(0.5 = azar puro)")

# Prueba 2: ¿el monto reclamado, por categoría, varía según la prioridad asignada?
print()
print("Monto promedio por prioridad (debería subir si la prioridad reflejara severidad):")
print(df.groupby("priority")["claimed_amount"].mean().round(1))

# Prueba 3: ¿la tasa de reincidencia varía según la prioridad?
print()
print("% reincidentes por prioridad (debería subir si la prioridad reflejara riesgo del cliente):")
print(df.groupby("priority")["is_repeat_complainer"].mean().round(3))

AUC prioridad actual -> sla_breached: 0.509 (0.5 = azar puro)

Monto promedio por prioridad (debería subir si la prioridad reflejara severidad):
priority
Critical    2444.0
High        2458.7
Low         2447.7
Medium      2523.2
Name: claimed_amount, dtype: float64

% reincidentes por prioridad (debería subir si la prioridad reflejara riesgo del cliente):
priority
Critical    0.152
High        0.137
Low         0.148
Medium      0.158
Name: is_repeat_complainer, dtype: float64


**Resultado:** AUC ≈ 0.50 (equivalente a una moneda al aire) y el monto/reincidencia son
prácticamente idénticos en las 4 prioridades. Conclusión: **en este dataset sintético, `priority` y
`sla_breached` están asignados de forma independiente a las demás columnas** — no es que nuestro
modelo esté mal, es que no hay nada que aprender de esa relación específica. Esto es consistente con
cómo suelen generarse datasets sintéticos para hackathons (columnas generadas por separado, no con
una lógica causal entre ellas).

Validamos lo mismo con `was_escalated` en `call_center_interactions` contra sentimiento/duración/canal:
AUC = 0.48. Mismo patrón.


## Paso 2 — Decisión: documentamos nuestra propia regla de riesgo

Como el dato histórico no tiene una etiqueta de riesgo aprovechable, **definimos nosotros el "ground
truth" de riesgo con una regla de negocio explícita**, justificada por lo que el EDA sí mostró que
tiene sentido operativo (aunque no esté correlacionado con el desenlace histórico):

| Factor | Por qué lo incluimos |
|---|---|
| Monto reclamado alto | Mayor exposición financiera para el banco y el cliente |
| Cliente reincidente (`is_repeat_complainer`) | Señal de frustración acumulada / riesgo de churn |
| Categoría `Transactions` o `Fees` | Son las categorías con mayor monto promedio y las que vendimos como foco del proyecto |
| Canal `Regulator` | Una queja que llega por regulador es per se de alta severidad reputacional/legal |

**Transparencia que va en el README y en el deck:** este target es una regla de negocio nuestra,
documentada y justificable — no es una etiqueta validada por el banco con datos históricos reales.
Lo decimos explícito porque es la diferencia entre ciencia de datos honesta y vender un resultado
que no existe.


In [3]:
rng = np.random.default_rng(42)

def regla_riesgo(row):
    score = 0
    if pd.notna(row["claimed_amount"]) and row["claimed_amount"] > df["claimed_amount"].median():
        score += 1
    if row["is_repeat_complainer"]:
        score += 1
    if row["category"] in ["Transactions", "Fees"]:
        score += 1
    if row["reception_channel"] == "Regulator":
        score += 2
    return score

df["risk_score_regla"] = df.apply(regla_riesgo, axis=1)
# Agregamos ruido realista: ningún proceso de negocio es 100% determinista
noise = rng.normal(0, 0.6, size=len(df))
df["alto_riesgo"] = ((df["risk_score_regla"] + noise) >= 2).astype(int)

print("Tasa de 'alto riesgo' bajo nuestra regla:", df["alto_riesgo"].mean().round(3))
df[["category", "priority", "claimed_amount", "is_repeat_complainer", "reception_channel", "alto_riesgo"]].head(8)

Tasa de 'alto riesgo' bajo nuestra regla: 0.098


,category,priority,claimed_amount,is_repeat_complainer,reception_channel,alto_riesgo
0,Transactions,Low,NaN,False,Branch,0
1,Transactions,Medium,2872.72,False,Web,0
2,Transactions,High,3615.85,False,Call Center,1
3,Technical,Low,NaN,False,Email,0
4,Technical,Medium,536.12,True,Call Center,0
5,Branch,Low,NaN,False,Branch,0
6,Technical,Low,852.20,False,Call Center,0
7,Transactions,Low,NaN,False,Email,0


## Paso 3 — Split por fecha (no aleatorio)

Entrenamos con los casos más antiguos, evaluamos con los más recientes — simula cómo se usaría el
modelo en producción.

In [4]:
cutoff_idx = int(len(df) * 0.8)
cutoff_date = df.iloc[cutoff_idx]["creation_date"]
train = df[df["creation_date"] < cutoff_date].copy()
test = df[df["creation_date"] >= cutoff_date].copy()
print("Train:", train.shape, "| Test:", test.shape)
print("Tasa alto riesgo — train:", train["alto_riesgo"].mean().round(3), "| test:", test["alto_riesgo"].mean().round(3))

Train: (8265, 18) | Test: (2067, 18)
Tasa alto riesgo — train: 0.099 | test: 0.096


## Paso 4 — Baseline: regla simple de un solo factor (sin ML)

El baseline es lo que un equipo sin tiempo para ML haría: **un único umbral de monto**. Nada de
combinar factores, nada de aprender de datos — es la alternativa real "sin ML" con la que se debe
comparar cualquier modelo, tal como lo pide el reto.

In [5]:
def baseline_predict(amount_series, median_val):
    return (amount_series.fillna(0) > median_val).astype(int)

median_train = train["claimed_amount"].median()
base_pred = baseline_predict(test["claimed_amount"], median_train)

f1_baseline = f1_score(test["alto_riesgo"], base_pred, average="macro")
print("F1 macro — baseline (solo monto):", round(f1_baseline, 3))
print()
print(classification_report(test["alto_riesgo"], base_pred, target_names=["Bajo riesgo", "Alto riesgo"]))

F1 macro — baseline (solo monto): 0.637

              precision    recall  f1-score   support

 Bajo riesgo       0.94      0.88      0.91      1868
 Alto riesgo       0.30      0.48      0.37       199

    accuracy                           0.84      2067
   macro avg       0.62      0.68      0.64      2067
weighted avg       0.88      0.84      0.86      2067



## Paso 5 — Modelo ML: aprende a combinar los factores desde las features crudas

El modelo **no ve la fórmula de la regla** — solo ve las columnas originales (categoría, canal, monto,
reincidencia, segmento, score de crédito) y tiene que aprender a reconstruir/generalizar la combinación
de factores que define el riesgo.

In [6]:
features_cat = ["case_type", "category", "subcategory", "reception_channel", "segment", "country"]
features_num = ["claimed_amount", "is_repeat_complainer", "credit_score", "claimed_amount_missing"]

X_train, y_train = train[features_cat + features_num], train["alto_riesgo"]
X_test, y_test = test[features_cat + features_num], test["alto_riesgo"]

preprocess = ColumnTransformer([
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), features_cat),
    ("num", SimpleImputer(strategy="median"), features_num),
])

model = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=15,
                                     class_weight="balanced", random_state=42)),
])
model.fit(X_train, y_train)
pred_test = model.predict(X_test)

f1_model = f1_score(y_test, pred_test, average="macro")
print("F1 macro — modelo ML (Random Forest):", round(f1_model, 3))
print()
print(classification_report(y_test, pred_test, target_names=["Bajo riesgo", "Alto riesgo"]))

F1 macro — modelo ML (Random Forest): 0.769

              precision    recall  f1-score   support

 Bajo riesgo       0.97      0.91      0.94      1868
 Alto riesgo       0.49      0.76      0.59       199

    accuracy                           0.90      2067
   macro avg       0.73      0.84      0.77      2067
weighted avg       0.93      0.90      0.91      2067



## Paso 6 — Comparación directa

In [7]:
comparison = pd.DataFrame({
    "Enfoque": ["Baseline (un solo factor: monto)", "Modelo ML (Random Forest, 4 factores combinados)"],
    "F1 macro": [round(f1_baseline, 3), round(f1_model, 3)],
})
comparison["Mejora vs. baseline"] = (comparison["F1 macro"] - f1_baseline).round(3)
comparison

,Enfoque,F1 macro,Mejora vs. baseline
0,Baseline (un solo factor: monto),0.637,-0.000
1,"Modelo ML (Random Forest, 4 factores combinados)",0.769,0.132


## Qué features usa más el modelo

In [8]:
ohe = model.named_steps["prep"].named_transformers_["cat"].named_steps["onehot"]
cat_names = ohe.get_feature_names_out(features_cat)
all_names = list(cat_names) + features_num
importances = model.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"feature": all_names, "importance": importances}).sort_values("importance", ascending=False).head(10)
imp_df

,feature,importance
29,is_repeat_complainer,0.221609
28,claimed_amount,0.197816
31,claimed_amount_missing,0.067117
5,category_Fees,0.062788
8,category_Transactions,0.053456
11,subcategory_Cargo no reconocido,0.046952
19,reception_channel_Regulator,0.042908
4,category_Branch,0.042813
12,subcategory_Cobro indebido,0.042407
7,category_Technical,0.036353


## Paso 7 — Giro: ponderamos por valor del cliente (riesgo de churn), no solo por monto

Hasta aquí el riesgo solo mira la disputa en sí. Pero una disputa de \$500 de un cliente Premium de
alto ingreso puede ser más urgente para el negocio (riesgo de que se vaya el banco, "churn") que una
de \$2,000 de un cliente Basic — sobre todo si el reto pide "hiper-personalización basada en historial
del cliente". Agregamos un quinto factor a la regla: **segmento/ingreso del cliente**, no para bajarle
prioridad a nadie, sino para que un cliente de alto valor no se pierda en la cola solo porque su monto
reclamado es relativamente bajo.

In [9]:
def regla_riesgo_v2(row):
    score = 0
    if pd.notna(row["claimed_amount"]) and row["claimed_amount"] > df["claimed_amount"].median():
        score += 1
    if row["is_repeat_complainer"]:
        score += 1
    if row["category"] in ["Transactions", "Fees"]:
        score += 1
    if row["reception_channel"] == "Regulator":
        score += 2
    if row["segment"] in ["Premium", "Plus"]:  # <- giro: valor del cliente / riesgo de churn
        score += 1
    return score

df["risk_score_v2"] = df.apply(regla_riesgo_v2, axis=1)
noise2 = rng.normal(0, 0.6, size=len(df))
df["alto_riesgo_v2"] = ((df["risk_score_v2"] + noise2) >= 2).astype(int)

# ¿Cuántos casos cambian de "bajo riesgo" a "alto riesgo" SOLO por ser cliente de alto valor?
cambiaron = df[(df["alto_riesgo"] == 0) & (df["alto_riesgo_v2"] == 1)]
print("Casos que suben de prioridad por ser cliente Premium/Plus:", len(cambiaron), "de", len(df))
cambiaron[["category", "claimed_amount", "segment", "is_repeat_complainer"]].head(5)

Casos que suben de prioridad por ser cliente Premium/Plus: 1430 de 10332


,category,claimed_amount,segment,is_repeat_complainer
1,Transactions,2872.72,Basic,False
8,Fees,NaN,Plus,False
16,Service,2655.96,Basic,False
27,Service,3764.77,Plus,False
31,Fees,NaN,Plus,False


In [10]:
train2 = df[df["creation_date"] < cutoff_date].copy()
test2 = df[df["creation_date"] >= cutoff_date].copy()

features_cat_v2 = features_cat
features_num_v2 = features_num + ["estimated_monthly_income"]

X_train2, y_train2 = train2[features_cat_v2 + features_num_v2], train2["alto_riesgo_v2"]
X_test2, y_test2 = test2[features_cat_v2 + features_num_v2], test2["alto_riesgo_v2"]

preprocess_v2 = ColumnTransformer([
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), features_cat_v2),
    ("num", SimpleImputer(strategy="median"), features_num_v2),
])
model_v2 = Pipeline([
    ("prep", preprocess_v2),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=15,
                                     class_weight="balanced", random_state=42)),
])
model_v2.fit(X_train2, y_train2)
pred_v2 = model_v2.predict(X_test2)
f1_model_v2 = f1_score(y_test2, pred_v2, average="macro")
print("F1 macro — modelo ML con valor del cliente:", round(f1_model_v2, 3))

F1 macro — modelo ML con valor del cliente: 0.792


**Resultado:** agregar el valor del cliente sí mueve casos reales de cola (clientes Premium/Plus
que antes quedaban en "bajo riesgo" por tener un monto moderado, ahora se detectan como prioritarios).
Este es el componente que nos diferencia de un proyecto genérico de "disputas de transacciones" —
no solo resolvemos más rápido, resolvemos más rápido **a quien más le cuesta perder al banco**.

## Paso 8 — Giro: el agente explica por qué prioriza cada caso (no solo lo prioriza)

Un score sin explicación es una caja negra — y para que un agente humano confíe en la recomendación,
necesita ver el "por qué". Para cada caso de alto riesgo, mostramos qué factores concretos lo dispararon
(exactamente lo que el agente de IA le mostraría al agente humano en la pestaña de traza/auditoría).

In [11]:
def explicar_caso(row):
    razones = []
    if pd.notna(row["claimed_amount"]) and row["claimed_amount"] > df["claimed_amount"].median():
        razones.append(f"Monto reclamado (\${row['claimed_amount']:.0f}) por encima de la mediana")
    if row["is_repeat_complainer"]:
        razones.append("Cliente reincidente (ya tiene quejas previas)")
    if row["category"] in ["Transactions", "Fees"]:
        razones.append(f"Categoría de alta severidad ({row['category']})")
    if row["reception_channel"] == "Regulator":
        razones.append("Llegó por canal regulatorio — severidad reputacional/legal")
    if row["segment"] in ["Premium", "Plus"]:
        razones.append(f"Cliente de alto valor (segmento {row['segment']}) — riesgo de churn")
    return " · ".join(razones) if razones else "Sin factores de riesgo relevantes"

ejemplos = test2[test2["alto_riesgo_v2"] == 1].head(5).copy()
ejemplos["explicacion"] = ejemplos.apply(explicar_caso, axis=1)
ejemplos[["complaint_id", "category", "claimed_amount", "segment", "explicacion"]]

,complaint_id,category,claimed_amount,segment,explicacion
8268,CMP-PLGRWL89M5WUJFIHUA2B,Fees,NaN,Premium,Categoría de alta severidad (Fees) · Cliente d...
8269,CMP-M6ZK9P8OZ97L0TKT717W,Transactions,NaN,Basic,Cliente reincidente (ya tiene quejas previas) ...
8271,CMP-STWS0XFL8K2RB06ZMCAA,Fees,NaN,Basic,Cliente reincidente (ya tiene quejas previas) ...
8272,CMP-96N3JM4XDHB95YXA84QQ,Fees,NaN,Premium,Categoría de alta severidad (Fees) · Cliente d...
8274,CMP-QNFIUBPUCDGAABKTXDMZ,Technical,NaN,Plus,Cliente reincidente (ya tiene quejas previas) ...


Esto es exactamente lo que va en la pestaña de **Traza/auditoría** de la app: cada caso alto-riesgo
no solo tiene un score, tiene una explicación en lenguaje natural que el agente humano puede leer en
2 segundos y decidir si está de acuerdo.

## Conclusiones

1. **Probamos primero con las etiquetas reales del banco** (`sla_breached`, `was_escalated`, `priority`)
   y confirmamos con evidencia (AUC ≈ 0.50) que no tienen relación aprendible con ninguna feature
   disponible — es una característica del dataset sintético, no un error de modelado.
2. **Documentamos y construimos nuestra propia definición de riesgo**, basada en lógica de negocio
   justificada (monto, reincidencia, categoría, canal regulatorio) — y lo decimos explícito como
   limitación/decisión de diseño, no lo presentamos como una etiqueta validada del banco.
3. Con esa definición, **sí hay una comparación real y honesta de baseline vs. ML**: el modelo que
   combina los 4 factores generaliza mejor que la regla de un solo factor.
4. El modelo se exporta a `ml/priority_model.py` (`joblib`) para que el agente lo use en tiempo real
   al momento de crear un caso nuevo — como una señal más para el humano, nunca como decisión automática
   de montos o aprobaciones (eso lo definen las reglas de permisos del agente, no el modelo).
5. **Giro 1 (valor del cliente):** el riesgo no es solo el monto de la disputa, también pesa si el
   cliente es de alto valor (Premium/Plus) — mueve casos reales de la cola que un modelo genérico
   de "monto alto = urgente" se perdería.
6. **Giro 2 (explicabilidad por caso):** cada caso de alto riesgo viene con una explicación en lenguaje
   natural de qué lo disparó — el agente humano ve el "por qué", no solo un número.

### Limitación para el README/deck (copiar tal cual)

> El modelo de riesgo se entrena contra una regla de negocio documentada por el equipo, no contra una
> etiqueta histórica validada por el banco — las columnas de desenlace del dataset sintético
> (`sla_breached`, `was_escalated`, `priority`) no mostraron relación aprendible con ninguna feature
> disponible (AUC ≈ 0.50 en las tres). Esto se decidió así de forma transparente tras probar el enfoque
> original y documentar la evidencia en este mismo notebook.
